# Ollama GPU Server on Google Colab (Receipt AI)

Questo notebook permette di eseguire **Ollama su GPU T4 gratuita** ed esporlo al tuo backend locale tramite **Cloudflare Tunnel**.

### Istruzioni rapide
1. Vai su **Runtime -> Change runtime type** e seleziona **T4 GPU**.
2. Scegli il modello nella cella 3 (es. `gemma4:e2b`).
3. Esegui le celle in ordine (`Run all`).
4. Copia l'URL `https://....trycloudflare.com` stampato dall'ultima cella e incollalo nel tuo file `.env`:
   ```env
   OLLAMA_URL=https://xxxx-xxxx.trycloudflare.com
   OLLAMA_MODEL=gemma4:e2b
   ```
5. Riavvia il backend locale (`uvicorn`).

In [ ]:
# 1. Verifica che la GPU T4 sia attiva
!nvidia-smi

In [ ]:
# 2. Installa zstd, pciutils (lspci per far rilevare la GPU a Ollama), Ollama e cloudflared
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!rm -f cloudflared-linux-amd64.deb

In [ ]:
#@title 3. Avvia Ollama e scarica il modello
MODEL_NAME = "gemma4:e2b" #@param {type:"string"}
MODEL_NAME = MODEL_NAME.strip() or "gemma4:e2b"

import os, subprocess, time, urllib.request

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"

subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)

# Attendi che Ollama sia pronto
for _ in range(20):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama attivo!")
        break
    except Exception:
        time.sleep(1)

print(f"Download del modello {MODEL_NAME}...")
!ollama pull {MODEL_NAME}

In [ ]:
# 4. Avvia il tunnel pubblico HTTPS (lascia questa cella in esecuzione!)
import re, subprocess

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Avvio tunnel Cloudflare...")
for line in proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print("\n" + "=" * 60)
        print(f"COPIA NEL TUO .env:\nOLLAMA_URL={url}")
        print("=" * 60 + "\n")
    if "ERR" in line or "Registered tunnel" in line:
        print(line.strip())